# Módulos

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Lectura

In [ ]:
p = Path(os.getcwd())
p = p.parents[2]
p1 = p / 'data' / 'processed' / 'precios.parquet'

df = pd.read_parquet(p1)

# Procesamiento

Drop de columnas que no se usarán

In [ ]:
df.drop(columns=['id', 'name', 'v_resnet', 'v_convnext', 'price_overview'], inplace=True)
df.head(2)

Release Year -> Label Encoding

In [ ]:
le = LabelEncoder()
df['release_year'] = df['release_year'].apply(lambda x : int(x))
df.head(2)

In [ ]:
df.to_parquet('simple_prices.parquet')

## Reducir columnas -> Transformaciones

Multiplayer

In [ ]:
multiplayer_columns = ['Multi-player', 'Online Co-op', 'Online PvP', 'PvP', 'Shared/Split Screen', 'Remote Play Together', 'Co-op']

df["has_multiplayer"] = df[multiplayer_columns].max(axis=1)
df["num_multiplayer_modes"] = df[multiplayer_columns].sum(axis=1)
df.drop(columns=multiplayer_columns, inplace=True)
df.head(2)

Steam Functions

In [ ]:

steam_functions = [ 'Steam Achievements', 'Steam Cloud', 'Steam Leaderboards', 'Steam Trading Cards']

def steam_category(row):
    total = row[steam_functions].sum()
    if total == 0: # No tiene
        return 0
    elif total == len(steam_functions): # Tiene Todas
        return 1
    else: # Tiene Algunas
        return 2

df["steam_features"] = df.apply(steam_category, axis=1)
df["num_steam_features"] = df[steam_functions].sum(axis=1)

df.drop(columns=steam_functions, inplace=True)
df.head(2)


Controller Support

In [ ]:
controller_columns = ['Full controller support', 'Partial Controller Support']

def controller_map(row):
    if row['Full controller support'] == 1:
        return 0
    elif row['Partial Controller Support'] == 1:
        return 1
    else:
        return 2

df["controller_support"] = df.apply(controller_map, axis=1)

df.drop(columns=controller_columns, inplace=True)
df.head(2)


Géneros

In [ ]:
genre_cols = [
    'Action','Adventure','Casual',
    'Indie','RPG','Simulation','Strategy'
]

df["genres"] = df[genre_cols].apply(
    lambda row: "|".join([col for col in genre_cols if row[col] == 1]),
    axis=1
)

df["genres"] = df["genres"].replace("", "none")

df.head(2)

In [ ]:
df['genres'].value_counts()

In [ ]:
top_genres = df["genres"].value_counts().nlargest(50).index
df["genres"] = df["genres"].where(df["genres"].isin(top_genres), "other")

genre_cols = ['Action','Adventure','Casual','Early Access','Indie','RPG','Simulation','Strategy']
df["num_genres"] = df[genre_cols].sum(axis=1)

df.head(5)

In [ ]:

genre_cols = ['Action','Adventure','Casual','Early Access','Indie','RPG','Simulation','Strategy']

df.drop(columns=genre_cols, inplace=True)
df.head(5)

In [ ]:
df.drop(columns=['Custom Volume Controls', 'Playable without Timed Input'], inplace=True)

In [ ]:
df.to_parquet('prices_reduced.parquet')